# Demo — Three-Identity Separation

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kpassoubady/bedrock-companion/blob/main/day1/demos/demo-identity-separation/demo-identity-separation.ipynb)

This notebook is a follow-along demo.

Day 1 — Block 3: Securing the Agent (Identity and Least Privilege)

Demonstrates the difference between caller identity (human), orchestration
identity (backend), and workload identity (agent). Explains why passing the
caller context via runtimeUserId is critical for zero-trust and attribution.

In [ ]:
# Install required dependencies
!pip install boto3 --quiet

In [ ]:
import json
import uuid

def print_section(title: str):
    print(f"\n{'=' * 60}")
    print(f"  {title}")
    print(f"{'=' * 60}")

def show_caller_identity():
    """1. Caller Identity (The Human / End User)"""
    print_section("1. Caller Identity (The Human)")

    caller = {
        "type": "Human User",
        "authenticated_by": "Amazon Cognito / Okta",
        "identity_id": "user-finance-881",
        "department": "Finance",
    }
    print(f"  {json.dumps(caller, indent=2)}")
    print("\n  - Evaluated at the edge (API Gateway)")
    print("  - Cannot be trusted blindly by downstream services")

def show_orchestration_identity():
    """2. Orchestration Identity (The Backend Service)"""
    print_section("2. Orchestration Identity (The App Backend)")

    backend = {
        "type": "Backend Service",
        "role": "arn:aws:iam::111122223333:role/ChatbotBackendRole",
        "permissions": ["bedrock-agentcore:InvokeAgentRuntime"],
    }
    print(f"  {json.dumps(backend, indent=2)}")
    print("\n  - The IAM role assigned to your Lambda/ECS container")
    print("  - Responsible for calling the AgentCore API")
    print("  - MUST pass the caller context via runtimeUserId")

def show_workload_identity():
    """3. Workload Identity (The Agent)"""
    print_section("3. Workload Identity (The Agent)")

    agent = {
        "type": "Agent Runtime",
        "role": "arn:aws:iam::111122223333:role/FinanceAgentExecutionRole",
        "permissions": ["bedrock-agentcore-gateway:InvokeTool"],
    }
    print(f"  {json.dumps(agent, indent=2)}")
    print("\n  - The IAM role assigned to the agent microVM")
    print("  - Cannot access resources outside its strict allowlist")

def show_invocation_pattern():
    """Bring it all together in the invocation pattern."""
    print_section("Bringing It Together: The Invocation Pattern")

    print("  When the backend calls AgentCore Runtime, it MUST include the caller:\n")

    invocation = {
        "api": "InvokeAgentRuntime",
        "agentRuntimeArn": "arn:aws:bedrock-agentcore:...",
        "runtimeSessionId": f"session-{str(uuid.uuid4())[:8]}",
        "runtimeUserId": "user-finance-881",  # The Caller Identity
    }
    print(f"  {json.dumps(invocation, indent=2)}")
    print("\n  Why is runtimeUserId critical?")
    print("  1. It proves to the audit log (CloudTrail) WHO initiated the action.")
    print("  2. Gateway can evaluate fine-grained IAM policies (e.g., 'Only finance users')")
    print("  3. Memory partitions state so User A cannot see User B's history.")

def main():
    print("Three-Identity Separation — Instructor Demo\n")
    show_caller_identity()
    show_orchestration_identity()
    show_workload_identity()
    show_invocation_pattern()

    print_section("Key Takeaways")
    print("  1. Never blend the caller, the backend, and the agent into one role.")
    print("  2. The agent is a separate workload with its own identity.")
    print("  3. runtimeUserId is the bridge that carries human context to the agent.")

if __name__ == "__main__":
    main()
